# Train Isolation Forest for GPS-Derived Features

This notebook mirrors `tools/dt_ids/train_isolation_forest.py`.

It loads the by-run split CSV files, excludes meta columns, trains a normal-only Isolation Forest, computes anomaly scores, and optionally saves the model bundle.

In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

In [ ]:
TRAIN_PATH = Path("data/splits/normal_gps_features_train.csv")
VAL_PATH = Path("data/splits/normal_gps_features_val.csv")
TEST_PATH = Path("data/splits/normal_gps_features_test.csv")
OUT_DIR = Path("data/models/iforest_gps_v1_notebook")

IGNORE_COLS = ["t", "mission_id", "split", "label"]
RANDOM_STATE = 42
N_ESTIMATORS = 200
MAX_SAMPLES = "auto"
CONTAMINATION = 0.01
THRESHOLD_QUANTILE = 0.99

OUT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_PATH, VAL_PATH, TEST_PATH, OUT_DIR

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("train shape:", train_df.shape)
print("val shape:", val_df.shape)
print("test shape:", test_df.shape)
train_df.head()

In [ ]:
feature_cols = [
    col
    for col in train_df.columns
    if col not in IGNORE_COLS and pd.api.types.is_numeric_dtype(train_df[col])
]

train_df = train_df.dropna(subset=feature_cols).reset_index(drop=True)
val_df = val_df.dropna(subset=feature_cols).reset_index(drop=True)
test_df = test_df.dropna(subset=feature_cols).reset_index(drop=True)

X_train = train_df[feature_cols].to_numpy()
X_val = val_df[feature_cols].to_numpy()
X_test = test_df[feature_cols].to_numpy()

print("num_features:", len(feature_cols))
feature_cols

In [ ]:
model = IsolationForest(
    n_estimators=N_ESTIMATORS,
    max_samples=MAX_SAMPLES,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
model.fit(X_train)

train_score = -model.decision_function(X_train)
val_score = -model.decision_function(X_val)
test_score = -model.decision_function(X_test)

threshold = float(np.quantile(train_score, THRESHOLD_QUANTILE))
threshold

In [ ]:
def summarize_split(split_name, df, score, threshold):
    flagged = score >= threshold
    summary = {
        "split": split_name,
        "num_rows": int(len(df)),
        "num_flagged": int(flagged.sum()),
        "flagged_ratio": float(flagged.mean()),
        "score_mean": float(np.mean(score)),
        "score_std": float(np.std(score)),
        "score_p95": float(np.quantile(score, 0.95)),
        "score_p99": float(np.quantile(score, 0.99)),
        "score_max": float(np.max(score)),
        "threshold": float(threshold),
    }
    if "mission_id" in df.columns:
        per_mission = (
            pd.DataFrame({"mission_id": df["mission_id"], "flagged": flagged.astype(int)})
            .groupby("mission_id", as_index=False)["flagged"]
            .mean()
            .rename(columns={"flagged": "flagged_ratio"})
        )
        summary["mission_flagged_ratio_mean"] = float(per_mission["flagged_ratio"].mean())
        summary["mission_flagged_ratio_max"] = float(per_mission["flagged_ratio"].max())
    return summary, flagged

train_summary, train_flagged = summarize_split("train", train_df, train_score, threshold)
val_summary, val_flagged = summarize_split("val", val_df, val_score, threshold)
test_summary, test_flagged = summarize_split("test", test_df, test_score, threshold)

summary_df = pd.DataFrame([train_summary, val_summary, test_summary])
summary_df

In [ ]:
model_bundle = {
    "model": model,
    "feature_columns": feature_cols,
    "ignore_columns": IGNORE_COLS,
    "threshold": threshold,
    "train_path": str(TRAIN_PATH.resolve()),
    "val_path": str(VAL_PATH.resolve()),
    "test_path": str(TEST_PATH.resolve()),
}

joblib.dump(model_bundle, OUT_DIR / "isolation_forest.joblib")
(OUT_DIR / "feature_columns.json").write_text(json.dumps(feature_cols, indent=2), encoding="utf-8")
(OUT_DIR / "training_config.json").write_text(
    json.dumps(
        {
            "ignore_cols": IGNORE_COLS,
            "random_state": RANDOM_STATE,
            "n_estimators": N_ESTIMATORS,
            "max_samples": MAX_SAMPLES,
            "contamination": CONTAMINATION,
            "threshold_quantile": THRESHOLD_QUANTILE,
            "threshold": threshold,
        },
        indent=2,
    ),
    encoding="utf-8",
)
summary_df.to_csv(OUT_DIR / "score_summary.csv", index=False)

train_scored = train_df.copy()
train_scored["anomaly_score"] = train_score
train_scored["is_flagged"] = train_flagged.astype(int)

val_scored = val_df.copy()
val_scored["anomaly_score"] = val_score
val_scored["is_flagged"] = val_flagged.astype(int)

test_scored = test_df.copy()
test_scored["anomaly_score"] = test_score
test_scored["is_flagged"] = test_flagged.astype(int)

train_scored.to_csv(OUT_DIR / "train_scored.csv", index=False)
val_scored.to_csv(OUT_DIR / "val_scored.csv", index=False)
test_scored.to_csv(OUT_DIR / "test_scored.csv", index=False)

summary_df

In [ ]:
test_scored.sort_values("anomaly_score", ascending=False).head(20)